# 🎸 Karplus-Strong Synthesis Explorer

Interactive comparison of four differentiable implementations against the NumPy oracle.

**Controls:** Adjust synthesis parameters, set an LFO with routing checkboxes, then hit **Generate**.  
**Output:** Five spectrograms side-by-side with audio playback for each.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

from synths.synth import Synth, SynthConfig

FS = 16000
DURATION = 4.0
NUM_SAMPLES = int(FS * DURATION)
NUM_FRAMES = 100
N_FFT = 2048
HOP = 512

/Users/pablotablasdepaula/PycharmProjects/DAFx26-Karplus/.pixi/envs/default/lib/python3.13/site-packages/philtorch/__init__.py:11: UserWarning: Custom extension not loaded.
  warnings.warn("Custom extension not loaded.")
/Users/pablotablasdepaula/PycharmProjects/DAFx26-Karplus/.pixi/envs/default/lib/python3.13/site-packages/torchlpc/__init__.py:23: UserWarning: Custom extension not loaded. Falling back to Numba implementation.
  warnings.warn("Custom extension not loaded. Falling back to Numba implementation.")


In [2]:
def apply_lfo(center, lfo, amplitude, lo, hi):
    """Apply LFO modulation, amplitude=1 means full symmetric swing within bounds."""
    max_dev = min(center - lo, hi - center)
    return np.clip(center + amplitude * max_dev * lfo, lo, hi)


def apply_lfo_log(center, lfo, amplitude, lo, hi):
    """Apply LFO in log-space (for f0)."""
    log_c, log_lo, log_hi = np.log(center), np.log(lo), np.log(hi)
    max_dev = min(log_c - log_lo, log_hi - log_c)
    return np.exp(np.clip(log_c + amplitude * max_dev * lfo, log_lo, log_hi))


def make_burst_gain(onset_freq, duration, num_frames, gain_values):
    burst_gain = np.zeros(num_frames)
    if onset_freq <= 0:
        burst_gain[0] = gain_values[0]
        return burst_gain
    period = num_frames / (onset_freq * duration)
    idx = 0.0
    while int(round(idx)) < num_frames:
        frame = int(round(idx))
        burst_gain[frame] = gain_values[frame]
        idx += period
    return burst_gain


IMPL_CONFIGS = [
    (False, False, "TD pluck + TD KS"),
    (False, True,  "TD pluck + FD KS"),
    (True,  False, "FD pluck + TD KS"),
    (True,  True,  "FD pluck + FD KS"),
]

In [3]:
# ── Parameter ranges (lo, hi) ──
RANGES = {
    'f0':              (55.0,  880.0),
    'burst_gain':      (0.01,  1.0),
    'dynamic_level':   (0.01,  1.0),
    'pluck_position':  (0.01,  1.0),
    'a1':              (0.01,  1.0),
    'decay':           (0.80,  1.0),
}

style  = {'description_width': '130px'}
layout = widgets.Layout(width='380px')

# ── Synthesis parameter sliders ──
sliders = {
    'f0': widgets.FloatLogSlider(
        value=220, base=2, min=np.log2(55), max=np.log2(880),
        step=0.01, description='f0 (Hz)',
        readout_format='.1f', style=style, layout=layout),
    'burst_gain': widgets.FloatSlider(
        value=0.5, min=0.01, max=1.0, step=0.01,
        description='Burst Gain', style=style, layout=layout),
    'dynamic_level': widgets.FloatSlider(
        value=0.5, min=0.01, max=1.0, step=0.01,
        description='Dynamic Level', style=style, layout=layout),
    'pluck_position': widgets.FloatSlider(
        value=0.5, min=0.00, max=1.0, step=0.01,
        description='Pluck Position', style=style, layout=layout),
    'a1': widgets.FloatSlider(
        value=0.5, min=0.00, max=1.0, step=0.01,
        description='a1 (Loop Filter)', style=style, layout=layout),
    'decay': widgets.FloatSlider(
        value=0.995, min=0.90, max=1.0, step=0.001,
        description='Decay', readout_format='.3f', style=style, layout=layout),
}

onset_slider = widgets.FloatSlider(
    value=2.0, min=0.25, max=16.0, step=0.25,
    description='Onset Freq (Hz)', style=style, layout=layout)

# ── LFO controls ──
lfo_freq_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=10.0, step=0.1,
    description='LFO Freq (Hz)', style=style, layout=layout)
lfo_amp_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.01,
    description='LFO Amplitude', style=style, layout=layout)

# ── LFO routing checkboxes ──
cb_layout = widgets.Layout(width='130px')
checkboxes = {
    name: widgets.Checkbox(value=False, description=name, layout=cb_layout, indent=False)
    for name in RANGES
}

# ── Generate button ──
gen_btn = widgets.Button(
    description='\U0001f3b8 Generate', button_style='primary',
    layout=widgets.Layout(width='220px', height='40px'))

# ── Output area ──
out = widgets.Output()


def on_generate(_):
    with out:
        clear_output(wait=True)

        # Build LFO waveform over frames
        t_frames = np.linspace(0, DURATION, NUM_FRAMES, endpoint=False)
        lfo = np.sin(2.0 * np.pi * lfo_freq_slider.value * t_frames)
        amp = lfo_amp_slider.value

        params = {}
        for name, sl in sliders.items():
            if name == 'burst_gain':
                continue
            center = sl.value
            lo, hi = RANGES[name]
            if checkboxes[name].value and amp > 0:
                if name == 'f0':
                    vals = apply_lfo_log(center, lfo, amp, lo, hi)
                else:
                    vals = apply_lfo(center, lfo, amp, lo, hi)
            else:
                vals = np.full(NUM_FRAMES, center)
            params[name] = torch.tensor(vals, dtype=torch.float32).unsqueeze(0)

        gain_center = sliders['burst_gain'].value
        lo, hi = RANGES['burst_gain']
        if checkboxes['burst_gain'].value and amp > 0:
            gain_values = apply_lfo(gain_center, lfo, amp, lo, hi)
        else:
            gain_values = np.full(NUM_FRAMES, gain_center)
        burst_gain_np = make_burst_gain(onset_slider.value, DURATION, NUM_FRAMES, gain_values)
        params['burst_gain'] = torch.tensor(burst_gain_np, dtype=torch.float32).unsqueeze(0)

        # ── Run all 5 synthesisers ──
        results = {}

        # Oracle
        oracle_cfg = SynthConfig(
            num_samples=NUM_SAMPLES, fs=FS, device='cpu')
        oracle_model = Synth(oracle_cfg)
        results['Oracle'] = oracle_model.oracle_synth(params).squeeze(0).numpy()

        # Four differentiable implementations
        for freq_pluck, freq_ksa, label in IMPL_CONFIGS:
            cfg = SynthConfig(
                num_samples=NUM_SAMPLES, fs=FS, device='cpu',
                use_freq_pluck=freq_pluck, use_freq_ksa=freq_ksa)
            model = Synth(cfg)
            with torch.no_grad():
                results[label] = model(params).squeeze(0).numpy()

        # ── Plot spectrograms ──
        names = ['Oracle'] + [c[2] for c in IMPL_CONFIGS]
        fig, axes = plt.subplots(1, 5, figsize=(28, 4), sharey=True)
        fig.suptitle('Spectrogram Comparison', fontsize=14, y=1.02)

        vmin, vmax = None, None  # compute shared colour limits
        specs = []
        for name in names:
            S = np.abs(np.fft.rfft(
                np.lib.stride_tricks.sliding_window_view(
                    np.pad(results[name], (N_FFT // 2, N_FFT // 2)),
                    N_FFT)[::HOP] * np.hanning(N_FFT), axis=-1))
            S_db = 20 * np.log10(S.T + 1e-10)
            specs.append(S_db)
            if vmin is None:
                vmin, vmax = S_db.min(), S_db.max()
            else:
                vmin = min(vmin, S_db.min())
                vmax = max(vmax, S_db.max())

        for ax, name, S_db in zip(axes, names, specs):
            freqs = np.linspace(0, FS / 2, S_db.shape[0])
            times = np.linspace(0, DURATION, S_db.shape[1])
            ax.pcolormesh(times, freqs, S_db,
                          vmin=vmax - 80, vmax=vmax,
                          shading='auto', cmap='magma')
            ax.set_title(name, fontsize=10)
            ax.set_xlabel('Time (s)')
            ax.set_ylim(0, FS / 2)
        axes[0].set_ylabel('Frequency (Hz)')
        plt.tight_layout()
        plt.show()

        # ── Audio playback ──
        print()
        for name in names:
            y = results[name]
            y_norm = y / (np.abs(y).max() + 1e-10) * 0.9
            print(f"▶ {name}")
            display(Audio(y_norm, rate=FS))


gen_btn.on_click(on_generate)

# ── Layout ──
param_box = widgets.VBox([
    widgets.HTML('<h3>Synthesis Parameters</h3>'),
    *sliders.values(),
    onset_slider,
])

lfo_box = widgets.VBox([
    widgets.HTML('<h3>LFO</h3>'),
    lfo_freq_slider,
    lfo_amp_slider,
    widgets.HTML('<b>Route LFO to:</b>'),
    widgets.HBox(list(checkboxes.values())),
])

top = widgets.HBox([param_box, widgets.VBox([lfo_box, gen_btn])],
                    layout=widgets.Layout(gap='40px'))

display(widgets.VBox([top, out]))